AR is a proxy metric for feature complexity


Get best model, compute switching probability for every single gate, save to a spreadsheet/data structure that can be easily loaded in python.
We need to give David the best model and the file with the switching probabilities. 


Switching probability: which gates are most active
For the best model, for different inputs, calculate how many times a gate's output is 1. 


Individual gates vs All gates
Does the model tend to prefer XOR gates? AND gates? Possible research directions or end of methodology findings.


CALCULATE SWITCH PROBABILITY FOR EACH GATE, NOT COMBINED 
e.g. 
AND1 was activated 20 times
AND2 was activated 12 times
These are calculated separately


(Pseudocode)
For image in MNIST:
  Run image through best model for inference:
    For gate in model
      If the specific gate_output == 1
        all_gates_dictionary[index_for_that_gate] += 1
        
        
After populating the counts dictionary, divide each count by total # of MNIST images and that is your Switching Probability.

The following script outlines the implementation of the classes needed to train and test a DiffLogic model

It uses config.yaml to load the model(s) configurations. For more information settings this up, expand the cell below.

### **Instructions for Populating the YAML Configuration File**

The YAML file below defines the configuration for a model named "agent", though several models can be added and named accordingly.

#### 1. General Settings

- **`input_dim`**: This is the dimensionality of the input data. Set this to the number of features in your input.
- **`output_size`**: This is the size of the output layer. Set this to the number of classes or outputs your model is expected to produce.
- **`tau`**: This is a hyperparameter that can be adjusted depending on your specific application. It might be related to time constants or smoothing parameters.
- **`learning_rate`**: This defines the learning rate for the optimizer. Adjust this according to the convergence behavior of your model during training.

#### 2. Layer Configurations for DiffLogic

- **`layers_config`**: This section defines the layers in your DiffLogic model. Each layer has several important parameters:
  - **`in_dim`**: The input dimension for the layer.
  - **`out_dim`**: The output dimension for the layer.
  - **`device`**: Specifies where the computations will take place. Use `'cuda'` if you have a compatible GPU, otherwise use `'cpu'`.
  - **`implementation`**: This typically matches the device but can be set depending on the specific implementation details (e.g., optimized CUDA code).
  - **`connections`**: Defines the type of connections in the layer. `'random'` suggests a random connection scheme, but this can be adjusted as needed (e.g., `'fully_connected'`, `'sparse'`, etc.).
  - **`grad_factor`**: A scaling factor applied to gradients in this layer. This can be adjusted to control the learning dynamics for specific layers.

#### 3. Example Configuration

Here’s an example configuration file:

```yaml
agent:
  # General settings
  input_dim: 60  # Number of input features
  output_size: 3  # Number of output classes
  tau: 30  # Example hyperparameter for the agent
  learning_rate: 0.001  # Learning rate for the optimizer

  # Layer configurations for DiffLogic
  layers_config:
    LogicLayer1:
      in_dim: 60  # Input dimension for the first logic layer
      out_dim: 30  # Output dimension for the first logic layer
      device: 'cuda'  # Use 'cuda' for GPU or 'cpu' for CPU
      implementation: 'cuda'  # Same as device, indicates CUDA implementation
      connections: 'random'  # Type of connections in the layer
      grad_factor: 2  # Gradient scaling factor for this layer
    LogicLayer2:
      in_dim: 30  
      out_dim: 60  
      device: 'cuda'  
      implementation: 'cuda'  
      connections: 'random'
      grad_factor: 1  
    # You may declare additional logic layes here

  # Additional settings for other components can be added here
```

Ensure to add your `config.yaml` file in the root of your project

### **Dependencies you will need**

In [1]:
import pandas as pd
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from difflogic import LogicLayer, GroupSum
import numpy as np
import matplotlib.pyplot as plt
from torchviz import make_dot
import sys

### **DataProcessor Class**

In [2]:
class DataProcessor:
    def __init__(self, file_path, data_id='data_id', state_id='state_id', test_size=0.2, batch_size=32, random_state=42):
        """
        Initializes the DataProcessor, loads the data, processes it, and splits it into training and testing datasets.

        Args:
            file_path (str): Path to the CSV file containing the data.
            data_id (str): Column name for feature data.
            state_id (str): Column name for label data.
            test_size (float): Fraction of the data to be used as the test set.
            batch_size (int): Number of samples per batch.
            random_state (int): Seed for the random number generator.
        """
        self.file_path = file_path
        self.data_id = data_id
        self.state_id = state_id
        self.test_size = test_size
        self.batch_size = batch_size
        self.random_state = random_state
        self.train_loader, self.test_loader = self.load_and_split_data()
        
    def process_chunk(self, data, variable):
        """
        Processes a chunk of data and converts it into tensors. The processing differs based on the data type (features or labels).

        Args:
            data (pandas.Series): A series object containing the data to be processed.
            variable (bool): If False, process as feature data (X); if True, process as label data (y).

        Returns:
            List[torch.Tensor]: A list of tensors corresponding to the processed data.
        """
        tensors = []
        if not variable:
            data = data.apply(lambda x: x.replace('\n', '').replace('[', '').replace(']', '').replace('.', '').split())
            data = data.apply(lambda x: [float(num) for num in x])
            tensors = [torch.tensor(entry, dtype=torch.float64).view(20, 3) for entry in data]
        else:
            data = data.apply(lambda x: eval(x))
            tensors = [torch.tensor(entry) for entry in data]
        return tensors

    def data_conversion_csv(self, column_name, variable, chunksize=10000):
        """
        A generator function that reads data from a CSV in chunks and processes it.

        Args:
            column_name (str): The name of the column to read data from.
            variable (bool): Determines how the data should be processed (as features or labels).
            chunksize (int): Number of rows per chunk.

        Yields:
            Iterator over processed data tensors.
        """
        for chunk in pd.read_csv(self.file_path, usecols=[column_name], chunksize=chunksize):
            yield from self.process_chunk(chunk[column_name], variable)

    def load_and_split_data(self):
        """
        Loads and processes data from a CSV file into tensors for features and labels, then splits them into training and testing sets.

        Returns:
            tuple: A tuple containing the DataLoader for training and testing datasets.
        """
        X_data = list(self.data_conversion_csv(self.data_id, False))
        y_data = list(self.data_conversion_csv(self.state_id, True))
        X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=self.test_size, random_state=self.random_state)
        X_train_tensor, X_test_tensor = torch.stack(X_train), torch.stack(X_test)
        y_train_tensor, y_test_tensor = torch.stack(y_train), torch.stack(y_test)
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        
        # Create DataLoaders with the specified batch size
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        return train_loader, test_loader

### **DiffLogic Class**

In [3]:
class DiffLogic(nn.Module):
    def __init__(self, layers_config, output_size=3, tau=30):
        """
        Initializes the DiffLogic model with the specified layer configurations, output size, and temperature parameter.

        Args:
            layers_config (dict): Configuration for each logic layer, including dimensions, device, implementation, connections, and grad factor.
            output_size (int): The number of output groups.
            tau (int): Temperature parameter for the GroupSum operation.
        """
        super(DiffLogic, self).__init__()
        self.flatten = nn.Flatten()
        
        layers = []
        for layer_name, config in layers_config.items():
            layer = LogicLayer(
                in_dim=config['in_dim'],
                out_dim=config['out_dim'],
                device=config['device'],
                implementation=config['implementation'],
                connections=config['connections'],
                grad_factor=config['grad_factor']       
            )
            layers.append(layer)
            print(layer)
        
        self.logic_layers = nn.Sequential(*layers)
        
        self.group = GroupSum(k=output_size, tau=tau)
    
    def forward(self, x):
        """
        Forward pass of the DiffLogic model.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after processing through the logic layers and grouping operation.
        """
        # Move tensor to GPU
        if torch.cuda.is_available():
            x = x.to('cuda')          
        x = self.flatten(x)
        logits = self.logic_layers(x)
        group = self.group(logits)
        return group

### **Overall Model Class**

In [4]:
class Model(nn.Module):
    def __init__(self, cfg):
        """
        Initializes the Model, loads the DiffLogic model, sets up the optimizer, loss function, and other necessary configurations.

        Args:
            cfg (dict): Configuration dictionary containing model parameters such as layer configurations, output size, tau, and learning rate.
        """
        super(Model, self).__init__()
        
        layers_config = cfg["layers_config"]
        output_size = cfg["output_size"]
        tau = cfg["tau"]
                
        self.diff_logic_model = DiffLogic(layers_config, output_size=output_size, tau=tau)
        self.optimizer = torch.optim.Adam(self.diff_logic_model.parameters(), lr=cfg["learning_rate"])
        self.log_text = ""  # Initialize the logging string
        self.data_batch = []
        self.criterion = nn.MSELoss()

        self.inputs = ["data_id"]
        self.outputs = ["state_id"]
        self.dtrain = ["reward", "state_id", "state_id_label"]
        self.needs = list(np.unique(self.dtrain + self.inputs))

    def forward(self, prediction_dict):    
        """
        Forward pass to predict the model output.

        Args:
            prediction_dict (dict): Dictionary containing input data for making predictions.

        Returns:
            None: Updates the prediction_dict with the predicted output data.
        """
        input_data = prediction_dict["input_data"]
        output = self.diff_logic_model(input_data)
        self.log_text += f"State Prediction: {output}\n"
        prediction_dict["pred_output_data"] = output

    def loss(self, prediction, label):
        """
        Computes the loss between the predicted and true labels.

        Args:
            prediction (torch.Tensor): Predicted tensor output from the model.
            label (torch.Tensor): True labels tensor.

        Returns:
            torch.Tensor: The computed loss value.
        """
        return self.criterion(prediction, label)

    def step(self):
        """
        Updates the model parameters by performing a single optimization step.

        Returns:
            None
        """
        self.optimizer.step()

    def train(self, tdata, mem, batch_size):
        """
        Training method that returns the loss after receiving a batch of data.

        Args:
            tdata (any): Training data.
            mem (any): Memory buffer or additional data needed for training.
            batch_size (int): Size of the data batch for training.

        Returns:
            torch.Tensor: The loss value for the batch.
        """
        self.optimizer.zero_grad()
        loss = torch.tensor([0.0 for i in range(batch_size)])
        return loss

    def test_train(self, prediction_dict):
        """
        Training method for system development on a dummy dataset.

        Args:
            prediction_dict (dict): Dictionary containing predicted and true output data.

        Returns:
            torch.Tensor: The computed loss.
        """
        self.optimizer.zero_grad()
        loss = self.loss(prediction_dict["pred_output_data"], prediction_dict["true_output_data"]).flatten()  # Structure loss (DNN relies on down-stream prediction)  
        return loss

    def save(self, file_path):
        """
        Saves the model's state dictionary to the specified file path.

        Args:
            file_path (str): Path where the model will be saved.

        Returns:
            None
        """
        torch.save(self.diff_logic_model.state_dict(), file_path)
        self.log_text += f"Model saved to: {file_path}\n"

    def load(self, file_path):
        """
        Loads the model's state dictionary from the specified file path.

        Args:
            file_path (str): Path from which the model will be loaded.

        Returns:
            None
        """
        self.diff_logic_model.load_state_dict(torch.load(file_path))
        self.log_text += f"Model loaded from: {file_path}\n"

    def plot_loss(self, loss_history):
        """
        Plots the training loss over epochs.

        Args:
            loss_history (list): List containing the loss values for each epoch.

        Returns:
            None
        """
        plt.figure()
        plt.plot(loss_history, label='Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training Loss Over Epochs')
        plt.legend()
        plt.show()
        self.log_text += "Loss plot generated\n"

    def visualize_model(self):
        """
        Generates a visualization of the model using a dummy input.

        Returns:
            None
        """
        dummy_input = torch.randn(1, cfg["input_dim"])  # Adjust this as needed based on your input dimensions
        y = self.diff_logic_model(dummy_input)
        g = make_dot(y, params=dict(self.diff_logic_model.named_parameters()))
        g.view()
        self.log_text += "Model visualization generated\n"

    def get_log(self):
        """
        Retrieves the log text and clears the log after retrieval.

        Returns:
            str: The log text.
        """
        log_copy = self.log_text
        self.log_text = ""  # Clear the log after returning
        return log_copy

    def explain(self):
        """
        Placeholder function for explaining the model's decisions.

        Returns:
            None
        """
        return None

### **Testing the Overall Model Class functions**

In [6]:
# Testing the class
if __name__ == "__main__":

    from hydra import compose, initialize
    from omegaconf import OmegaConf

    # Load the config file and create the model 
    with initialize(version_base=None, config_path="", job_name="test_app"):
        cfg = compose(config_name="config")
    model = Model(cfg['agent'])
    model = model.to('cuda')
    
    print(model) # this is a list of models, we can use this to test MANY models at once
    # just add more models to the config.yaml file
    
    # Ask Stephen about code functionality below, I did not find it necessary
    #dataset_path = "/blue/woodard/share/skeptical_beings/training_datasets/schoolhouse_dataset_e2_2024.06.14.csv"    
    #forward_list = {iny: [] for iny in model.inputs + ["data_id_label"]}
    #train_list = {tiny: [] for tiny in model.dtrain if tiny not in model.outputs}
    
    dataset_path = "/blue/woodard/share/skeptical_beings/training_datasets/schoolhouse_dataset_e2_2024.06.14.csv"    
    dloader = DataProcessor(file_path=dataset_path)

    # Loads train and test datasets from dloader
    train_data, test_data = dloader.load_and_split_data()
    input_data = train_data.tensors[0] # gets the X variable (inputs) of the train data
    output_data = train_data.tensors[1] # gets the y variable (outputs) of the train data
    
    # puts tensors in the GPU before adding them to a dictionary
    if torch.cuda.is_available():
        input_data = input_data.to('cuda').float()
        output_data = output_data.to('cuda').float()
    
    prediction_dict = {"input_data": input_data, "true_output_data": output_data} # initialiazes a dictionary which will be used to store predictions

    
    print(input_data)
    print(f"... got data...")
    
    model.forward(prediction_dict)
    print(f"... forward() worked!")

    loss = model.test_train(prediction_dict)
    print(f"... test_train() worked!")

    loss.backward()
    print(f"... backward() worked!")

    model.step()
    print(f"... step() worked!")

    path = "./s_model.pth"
    model.save(path)
    print(f"... save() worked!")

    model.load(path)
    print(f"... load() worked!")

LogicLayer(60, 30, train)
LogicLayer(30, 60, train)
Model(
  (diff_logic_model): DiffLogic(
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (logic_layers): Sequential(
      (0): LogicLayer(60, 30, train)
      (1): LogicLayer(30, 60, train)
    )
    (group): GroupSum(k=3, tau=30)
  )
  (criterion): MSELoss()
)
tensor([[[0., 1., 1.],
         [2., 2., 2.],
         [2., 2., 2.],
         ...,
         [2., 3., 2.],
         [2., 1., 2.],
         [1., 2., 1.]],

        [[0., 0., 0.],
         [1., 3., 3.],
         [1., 1., 1.],
         ...,
         [1., 1., 2.],
         [1., 3., 2.],
         [2., 3., 1.]],

        [[0., 1., 1.],
         [1., 2., 2.],
         [1., 2., 2.],
         ...,
         [1., 2., 2.],
         [1., 2., 2.],
         [2., 1., 1.]],

        ...,

        [[0., 1., 1.],
         [2., 1., 1.],
         [3., 3., 3.],
         ...,
         [2., 2., 2.],
         [3., 2., 2.],
         [3., 1., 1.]],

        [[1., 0., 1.],
         [1., 2., 1.],
      